# 02 — Full Experiment: 21 Training Runs (PGR207 CIFAR-10 mid-term)

**Student IDs:** `<STUDENT_ID_1>`, `<STUDENT_ID_2>` *(fill in before submitting)*

Runs all **7 configurations x 3 seeds = 21 real training runs** on the full CIFAR-10 training
set, and evaluates each run once on the held-out 10,000-image test set. Results are appended
to `all_results.csv` **after every single run**, so a Colab disconnect partway through never
loses completed work — re-running this notebook automatically **skips runs already present**
in `all_results.csv` and only trains what's missing.

Only run this after `01_smoke_test.ipynb` has passed without errors.

**Fixed settings, identical across every configuration (per the OFAT design):**
- Epoch budget: `FULL_RUN_EPOCHS` = 25, early-stopping patience: `FULL_RUN_PATIENCE` = 7
  (validation loss). See the comment next to `FULL_RUN_EPOCHS` in `shared_pipeline.py` for
  the justification to reuse in the Methods section.
- Batch size: `BATCH_SIZE` = 128.
- Loss: cross-entropy. LR schedule: cosine annealing over the epoch budget.
- Data split: the same stratified 90/10 split cached in `val_split_indices.npz`
  (created by whichever notebook ran first).
- Seeds: 0, 1, 2 (`SEEDS` in `shared_pipeline.py`) — control weight init, data shuffling, and
  augmentation draws only; the validation split itself never changes.


## 1. Setup

In [ ]:
import shared_pipeline as sp

print("Device:", sp.DEVICE)
assert sp.DEVICE.type in ("cuda", "mps"), (
    "No GPU detected — 21 full runs on CPU would be extremely slow. "
    "On Colab: Runtime > Change runtime type > GPU."
)


In [ ]:
full_train, test_set = sp.load_raw_datasets()
train_idx, val_idx = sp.get_stratified_split(full_train.targets)
print(f"Train pool: {len(train_idx)}  Val: {len(val_idx)}  Test (held out): {len(test_set)}")


## 2. Build the run plan (7 configs x 3 seeds), skipping already-completed runs

If `all_results.csv` already contains some (config_id, seed) pairs — e.g. from a previous,
interrupted execution of this same notebook — those runs are skipped rather than repeated.


In [ ]:
RESULTS_CSV = sp.RESULTS_CSV_DEFAULT  # "all_results.csv"

completed = sp.get_completed_runs(RESULTS_CSV)
print(f"Already completed: {len(completed)} / {len(sp.CONFIGS) * len(sp.SEEDS)} runs")

run_plan = [
    (cid, cfg, seed)
    for cid, cfg in sp.CONFIGS.items()
    for seed in sp.SEEDS
    if (cid, seed) not in completed
]
print(f"Runs remaining this session: {len(run_plan)}")
for cid, cfg, seed in run_plan:
    print(f"  {cid} seed={seed} {cfg}")


## 3. Train

Each run: builds fresh data loaders for its augmentation strategy, builds a fresh model, trains
with early stopping on validation loss, restores the best-validation checkpoint, evaluates once
on the test set, and appends one row to `all_results.csv` immediately.

This cell can be safely re-run after an interruption — completed runs from the plan above are
already excluded.


In [ ]:
for cid, cfg, seed in run_plan:
    print(f"\n=== {cid} seed={seed} {cfg} ===")
    result = sp.run_training(
        config_id=cid,
        config=cfg,
        seed=seed,
        full_train=full_train,
        test_set=test_set,
        train_idx=train_idx,
        val_idx=val_idx,
        epochs=sp.FULL_RUN_EPOCHS,
        patience=sp.FULL_RUN_PATIENCE,
        batch_size=sp.BATCH_SIZE,
        subset_size=None,   # full training set
        verbose=True,
    )
    sp.append_result(result, csv_path=RESULTS_CSV)
    print(f"  -> saved. test_acc={result['test_accuracy']:.3f}  macro_f1={result['test_macro_f1']:.3f}  "
          f"epochs_run={result['epochs_run']}  early_stopped={result['early_stopped']}  "
          f"time={result['train_time_seconds']:.0f}s")

print("\nAll planned runs for this session are complete.")


## 4. Verify all 21 runs are present


In [ ]:
final_df = sp.load_results(RESULTS_CSV)
print(f"Total rows in {RESULTS_CSV}: {len(final_df)} (expected 21)")
final_df.groupby("config_id")["seed"].apply(list)


If this shows fewer than 21 rows (e.g. the runtime disconnected), just re-run the notebook
from the top — Section 2 will recompute the remaining run plan from `all_results.csv` and
Section 3 will only train what's missing.

Next: open `03_results_aggregation.ipynb` to compute mean +/- std per configuration and build
the per-factor summary tables and confusion matrices.
